In [ ]:
#실행 후 세션 재시작
#재시작 후에는 실행 X
!pip install "numpy<2" "opencv-python<4.10.0" ultralytics facenet-pytorch

In [ ]:
import os
import cv2
import torch
from torchvision import transforms
from PIL import Image
from ultralytics import YOLO
from IPython.display import display
from model import get_model


In [ ]:
# ==========================================
# [설정 변수]
MODEL_NAME = 'resnet'
SAVED_MODEL_PATH = '/content/best_model_resnet_augTrue_preFalse.pth'
IMAGE_PATH = '/content/my_photo.jpg' # 단체 사진 파일명

#===========================================
# [분류될 클래스 및 라벨 색상]
CLASSES = ['0_Infant', '1_Teen', '2_Adult', '3_Senior']

CLASS_COLORS = {
    0: (0, 165, 255),   # Infant: 주황색 (Orange)
    1: (0, 255, 255),   # Teen: 노란색 (Yellow)
    2: (255, 0, 0),     # Adult: 파란색 (Blue)
    3: (255, 0, 255)    # Senior: 보라색 (Purple)
}
# ==========================================

# 파일 존재 여부 확인
if not os.path.exists(IMAGE_PATH):
    print(f"❌ '{IMAGE_PATH}' 파일을 찾을 수 없습니다.")
else:
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"[*] 실행 환경: {device}")

    # 2. 모델 로드
    print("[*] YOLO 탐지 모델과 연령대 분석 모델을 불러오는 중입니다...")
    yolo_model = YOLO('yolov10l.pt')

    age_model = get_model(model_name=MODEL_NAME, num_classes=len(CLASSES), pretrained=False)
    age_model.load_state_dict(torch.load(SAVED_MODEL_PATH, map_location=device, weights_only=False))
    age_model.to(device)
    age_model.eval()

    # 3. 모델 입력용 전처리
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # 4. 이미지 로드 및 탐지 시작
    img_cv2 = cv2.imread(IMAGE_PATH)
    print("[*] 사진 속 모든 사람을 탐지하고 분석합니다. 잠시만 기다려주세요...")

    # YOLO 탐지
    results = yolo_model(img_cv2, conf=0.15, imgsz=960, verbose=False)
    boxes = results[0].boxes
    detect_count = 0

    # 5. 탐지된 객체 반복 처리
    for box in boxes:
        cls_id = int(box.cls[0].item())
        if cls_id != 0: # 사람이 아니면 건너뜀
            continue

        detect_count += 1
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

        # 이미지 밖으로 박스가 나가지 않게 조절
        h_img, w_img, _ = img_cv2.shape
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w_img, x2), min(h_img, y2)

        cropped_img_cv2 = img_cv2[y1:y2, x1:x2]
        if cropped_img_cv2.size == 0: continue

        # 연령대 분석을 위해 PIL 이미지로 변환
        cropped_img_rgb = cv2.cvtColor(cropped_img_cv2, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(cropped_img_rgb)

        img_tensor = transform(pil_img).unsqueeze(0).to(device)
        with torch.no_grad():
            outputs = age_model(img_tensor)
            _, predicted_idx = torch.max(outputs, 1)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)[0]
            confidence = probabilities[predicted_idx.item()] * 100

        # 라벨 생성
        predicted_class = CLASSES[predicted_idx.item()].split('_')[1]
        label = f"{predicted_class} ({confidence:.1f}%)"

        box_color = CLASS_COLORS[predicted_idx.item()]

        # 6. 원본 사진에 박스와 라벨 그리기 (가져온 색상 적용)
        cv2.rectangle(img_cv2, (x1, y1), (x2, y2), box_color, 2)
        (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(img_cv2, (x1, y1 - 25), (x1 + text_w, y1), box_color, -1)

        # 글씨는 어느 배경색에서나 잘 보이도록 검은색(0, 0, 0) 유지
        cv2.putText(img_cv2, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

    # 7. 주피터 노트북 화면에 결과 출력
    if detect_count > 0:
        print("\n" + "="*40)
        print(f" [분석 완료] 총 {detect_count}명의 연령대를 탐지했습니다!")
        print("="*40)

        # OpenCV 이미지를 주피터에서 보기 좋게 PIL 포맷으로 변환
        final_img_rgb = cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB)
        final_pil_img = Image.fromarray(final_img_rgb)

        # 노트북 셀 바로 아래에 이미지를 띄움
        display(final_pil_img)

        # 다운로드용으로 파일 저장
        final_pil_img.save("final_yolo_result.jpg")
        print("[*] 'final_yolo_result.jpg' 파일로도 저장되었습니다.")
    else:
        print("❌ 사진에서 사람을 찾지 못했습니다.")